# Multi-Episode Sequential Promotion Study (DSADS, 3 Rounds x B=4) — v2
Evaluates **sequential vocabulary growth** across 3 consecutive $B=4$ promotion episodes (total budget $= 12$) using `DSADS_EMBEDDINGS.npz`, `DSADS_WINDOWS.npz`, and `SOURCE_TRANSFER_BUNDLE.npz`.

Click **Runtime -> Run all** (runs on CPU in <1 minute).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json
from pathlib import Path
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist

PROJECT_DIR = Path("/content/drive/MyDrive/FedVocabHAR")
OUT_DIR = PROJECT_DIR / "artifacts" / "sequential_promotion"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def find_file(filename):
    matches = list(PROJECT_DIR.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"{filename} not found under {PROJECT_DIR}")
    return matches[0]

emb_npz = np.load(find_file("DSADS_EMBEDDINGS.npz"), allow_pickle=True)
win_npz = np.load(find_file("DSADS_WINDOWS.npz"), allow_pickle=True)
src_npz = np.load(find_file("SOURCE_TRANSFER_BUNDLE.npz"), allow_pickle=True)
print("DSADS_EMBEDDINGS keys:", emb_npz.files)
print("DSADS_WINDOWS keys:", win_npz.files)
print("SOURCE_TRANSFER_BUNDLE keys:", src_npz.files)

In [ ]:
def _pick(npz, keys):
    for k in keys:
        if k in npz.files:
            return npz[k]
    raise KeyError(f"None of {keys} in {npz.files}")

MAPPING = {
    1: "sitting", 2: "standing", 3: "lying", 4: "lying", 5: "stairs_up", 6: "stairs_down",
    7: "standing", 8: "moving_in_elevator", 9: "walking", 10: "walking", 11: "walking",
    12: "running", 13: "stepper", 14: "cross_trainer", 15: "cycling", 16: "cycling",
    17: "rowing", 18: "jumping", 19: "basketball"
}
KNOWN_NAMES = ["walking", "running", "sitting", "standing"]
NOVEL_NAMES = [
    "lying", "stairs_up", "stairs_down", "moving_in_elevator",
    "stepper", "cross_trainer", "cycling", "rowing", "jumping", "basketball"
]
TAU_LOC = 0.2090341925621032
TAU_AGREE = 0.8902801990509033

Z = emb_npz["embeddings"].astype(np.float64)
Z = Z / np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-12)
emb_wids = emb_npz["window_ids"].astype(np.int64)

win_wids = _pick(win_npz, ["window_ids", "opaque_ids", "ids"]).astype(np.int64)
win_subj = _pick(win_npz, ["subjects", "subject", "subject_ids"]).astype(int)
win_lbl_raw = _pick(win_npz, ["canonical_names", "activities", "activity", "raw_activities", "labels", "y"])

# Align DSADS_WINDOWS metadata to DSADS_EMBEDDINGS window_ids
wid_to_pos = {int(w): i for i, w in enumerate(win_wids)}
order = np.array([wid_to_pos[int(w)] for w in emb_wids], dtype=np.int64)
subjects = win_subj[order]
raw_vals = win_lbl_raw[order]
labels = np.array([MAPPING[int(v)] if str(v).isdigit() else str(v) for v in raw_vals])

P_known_arr = src_npz["recognition_prototypes"].astype(np.float64)
P_known_arr = P_known_arr / np.maximum(np.linalg.norm(P_known_arr, axis=1, keepdims=True), 1e-12)
P_known_cal = src_npz["calibration_prototypes"].astype(np.float64)
P_known_cal = P_known_cal / np.maximum(np.linalg.norm(P_known_cal, axis=1, keepdims=True), 1e-12)

known_proto = {name: P_known_arr[i] for i, name in enumerate(KNOWN_NAMES)}
cal_proto = {name: P_known_cal[i] for i, name in enumerate(KNOWN_NAMES)}
print("Aligned Z:", Z.shape, "| Subjects:", sorted(set(subjects.tolist())), "| Labels:", sorted(set(labels.tolist())))

In [ ]:
ACQ_SUBJ = [1, 2, 3, 4]
EVAL_SUBJ = [5, 6, 7, 8]

def eval_subject_mean(proto_dict):
    avail = list(proto_dict.keys())
    P = np.stack([proto_dict[k] for k in avail], axis=0)
    nov_subj, kn_subj = [], []
    for sid in EVAL_SUBJ:
        idx = np.flatnonzero(subjects == sid)
        sims = Z[idx] @ P.T
        preds = np.array([avail[i] for i in np.argmax(sims, axis=1)])
        y_s = labels[idx]
        def _f1(cls_list):
            vals = []
            for c in cls_list:
                tp = int(np.sum((y_s == c) & (preds == c)))
                fp = int(np.sum((y_s != c) & (preds == c)))
                fn = int(np.sum((y_s == c) & (preds != c)))
                d = 2 * tp + fp + fn
                vals.append((2.0 * tp / d) if d > 0 else 0.0)
            return float(np.mean(vals))
        nov_subj.append(_f1(NOVEL_NAMES))
        kn_subj.append(_f1(KNOWN_NAMES))
    return float(np.mean(nov_subj)), float(np.mean(kn_subj))

def build_round_candidates(cur_cal_proto, queried_ids):
    P_cal = np.stack(list(cur_cal_proto.values()), axis=0)
    clusters = []
    for sid in ACQ_SUBJ:
        idx = np.flatnonzero((subjects == sid) & (~np.isin(np.arange(len(Z)), list(queried_ids))))
        if len(idx) < 12:
            continue
        nov = 1.0 - np.max(Z[idx] @ P_cal.T, axis=1)
        top = idx[np.argsort(-nov)[:min(600, len(idx))]]
        dist_mat = pdist(Z[top], metric="cosine")
        cids = fcluster(linkage(dist_mat, method="average"), t=TAU_LOC, criterion="distance")
        for cid in np.unique(cids):
            members = top[cids == cid]
            if len(members) >= 12:
                sig = Z[members].mean(axis=0)
                sig = sig / max(np.linalg.norm(sig), 1e-12)
                medoid = int(members[np.argmax(Z[members] @ sig)])
                clusters.append({"client": sid, "size": len(members), "sig": sig, "medoid": medoid})
    best = {}
    for i, ca in enumerate(clusters):
        neigh = [(float(ca["sig"] @ cb["sig"]), j) for j, cb in enumerate(clusters) if cb["client"] != ca["client"] and float(ca["sig"] @ cb["sig"]) >= TAU_AGREE]
        if neigh:
            best[i] = max(neigh)[1]
    pairs = []
    for i, j in best.items():
        if i < j and best.get(j) == i:
            g = clusters[i]["sig"] + clusters[j]["sig"]
            g = g / max(np.linalg.norm(g), 1e-12)
            v = i if float(clusters[i]["sig"] @ g) >= float(clusters[j]["sig"] @ g) else j
            pairs.append((clusters[i]["size"] + clusters[j]["size"], clusters[v]["medoid"]))
    pairs.sort(key=lambda x: -x[0])
    return clusters, [m for _, m in pairs]

def run_sequential(policy, seed=0, rounds=3, budget=4):
    rec_proto = dict(known_proto)
    cal_p = dict(cal_proto)
    queried = set()
    rng = np.random.default_rng(seed)
    history = []
    for r in range(1, rounds + 1):
        clusters, mnn_medoids = build_round_candidates(cal_p, queried)
        if policy == "ranked_mnn":
            picks = mnn_medoids[:budget]
        else:
            all_meds = [c["medoid"] for c in clusters]
            if all_meds:
                sel_idx = rng.choice(len(all_meds), size=min(budget, len(all_meds)), replace=False)
                picks = [all_meds[i] for i in sel_idx]
            else:
                picks = []
        for med in picks:
            queried.add(med)
            lbl = labels[med]
            if lbl in NOVEL_NAMES and lbl not in rec_proto:
                rec_proto[lbl] = Z[med]
                cal_p[lbl] = Z[med]
        nov_f1, kn_f1 = eval_subject_mean(rec_proto)
        recovered = len([k for k in rec_proto if k in NOVEL_NAMES])
        history.append({
            "round": r,
            "queries_this_round": len(picks),
            "cum_queries": len(queried),
            "cum_recovered": recovered,
            "novel_f1": nov_f1,
            "known_f1": kn_f1,
        })
    return history

mnn_hist = run_sequential("ranked_mnn")
rand_runs = [run_sequential("random", seed=20261117 + s) for s in range(20)]
rand_hist = []
for r_idx in range(3):
    rand_hist.append({
        "round": r_idx + 1,
        "cum_queries": float(np.mean([h[r_idx]["cum_queries"] for h in rand_runs])),
        "cum_recovered": float(np.mean([h[r_idx]["cum_recovered"] for h in rand_runs])),
        "novel_f1": float(np.mean([h[r_idx]["novel_f1"] for h in rand_runs])),
        "known_f1": float(np.mean([h[r_idx]["known_f1"] for h in rand_runs])),
    })

res = {"ranked_mnn": mnn_hist, "random_20seed_mean": rand_hist}
(OUT_DIR / "dsads_sequential_3rounds.json").write_text(json.dumps(res, indent=2))
print("RESULTS:", json.dumps(res, indent=2))